# 🏏 IPL Win Predictor

**Goal:** during an IPL match, after every ball, predict each team's chance of winning — *"CSK 68% to win"* — just like the win-probability bar on TV.

This is a real-world project: nobody hands you a tidy table. You start from **raw ball-by-ball data** (JSON files, one per match) and build everything yourself.

**New skills in this project**
- Parsing raw **JSON event data** into tables
- **Feature engineering** from events: runs needed, balls left, required run rate…
- **Data leakage** — the silent killer of ML projects — and how to avoid it
- Splitting data **by match and by season**, not by row
- Predicting **probabilities** and checking they're honest (**calibration**, log loss, Brier score)
- Win-probability "worm" charts

⏱️ Two to three weeks. Go section by section. 
**How to run:** open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file. Run cells top to bottom with **Shift + Enter**.

## 1. Download the data

[Cricsheet](https://cricsheet.org) publishes free ball-by-ball data for every IPL match since 2008 — over 1,100 matches.

In [ ]:
import os, json, glob, zipfile, urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if not os.path.exists("ipl_json.zip"):
    urllib.request.urlretrieve("https://cricsheet.org/downloads/ipl_json.zip", "ipl_json.zip")
with zipfile.ZipFile("ipl_json.zip") as z:
    z.extractall("ipl_json")
files = sorted(glob.glob("ipl_json/*.json"))
print(len(files), "match files")

## 2. What does one match look like?
Each file is **JSON** — nested dictionaries and lists. Let's peek inside.

In [ ]:
with open(files[0]) as f:
    match = json.load(f)
print("top-level keys:", list(match.keys()))
info = match["info"]
print("date:", info["dates"][0], "| teams:", info["teams"], "| venue:", info.get("venue"))
print("outcome:", info["outcome"])
first_ball = match["innings"][0]["overs"][0]["deliveries"][0]
print("\nfirst ball:", json.dumps(first_ball, indent=2))

The structure is: `innings → overs → deliveries`. Each delivery has the batter, bowler, runs, and optionally `extras` (wides, no-balls…) and `wickets`.

⚠️ **Cricket detail:** wides and no-balls don't count as one of the over's 6 legal balls. We need to track that to know how many balls are left.

## 3. Parse every match into tables

In [ ]:
RENAME = {   # franchises that changed their names
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}
SHORT = {
    "Chennai Super Kings": "CSK", "Mumbai Indians": "MI", "Royal Challengers Bengaluru": "RCB",
    "Kolkata Knight Riders": "KKR", "Delhi Capitals": "DC", "Punjab Kings": "PBKS",
    "Rajasthan Royals": "RR", "Sunrisers Hyderabad": "SRH", "Gujarat Titans": "GT",
    "Lucknow Super Giants": "LSG", "Deccan Chargers": "DCH", "Pune Warriors": "PWI",
    "Rising Pune Supergiant": "RPS", "Gujarat Lions": "GL", "Kochi Tuskers Kerala": "KTK",
}
def short(team):
    return SHORT.get(team, team)

match_rows, ball_rows = [], []
for path in files:
    with open(path) as f:
        m = json.load(f)
    info = m["info"]
    mid = os.path.basename(path).replace(".json", "")
    outcome = info.get("outcome", {})
    teams = [RENAME.get(t, t) for t in info["teams"]]
    winner = RENAME.get(outcome.get("winner"), outcome.get("winner"))
    main_innings = [inn for inn in m.get("innings", []) if not inn.get("super_over")]
    match_rows.append({
        "match_id": mid,
        "date": pd.to_datetime(info["dates"][0]),
        "venue": info.get("venue"),
        "team1": teams[0], "team2": teams[1],
        "winner": winner,
        "result": outcome.get("result", "normal"),       # "tie" / "no result" / normal
        "method": outcome.get("method"),                  # e.g. "D/L" for rain-affected
        "overs": info.get("overs", 20),
        "n_innings": len(main_innings),
    })
    for inn_no, inn in enumerate(main_innings, start=1):
        batting = RENAME.get(inn["team"], inn["team"])
        for over in inn["overs"]:
            for d in over["deliveries"]:
                extras = d.get("extras", {})
                ball_rows.append({
                    "match_id": mid,
                    "innings": inn_no,
                    "batting_team": batting,
                    "over": over["over"],
                    "runs": d["runs"]["total"],
                    "wicket": len(d.get("wickets", [])),
                    "legal": int("wides" not in extras and "noballs" not in extras),
                })

matches = pd.DataFrame(match_rows)
balls = pd.DataFrame(ball_rows)
matches["season"] = matches["date"].dt.year
print(matches.shape, balls.shape)
matches.tail()

### Clean up
We keep only normal, full 20-over matches with a winner — no washouts, ties decided by super over, or rain-shortened (D/L) games.

In [ ]:
ok = (
    (matches["result"] == "normal") & matches["method"].isna() &
    (matches["overs"] == 20) & (matches["n_innings"] == 2) & matches["winner"].notna()
)
matches = matches[ok].reset_index(drop=True)
balls = balls[balls["match_id"].isin(matches["match_id"])].reset_index(drop=True)
print(len(matches), "clean matches from", matches["season"].min(), "to", matches["season"].max())

## 4. Explore

In [ ]:
totals = balls.groupby(["match_id", "innings"])["runs"].sum().unstack()
totals.columns = ["first_innings", "second_innings"]
by_season = totals.join(matches.set_index("match_id")["season"]).groupby("season")["first_innings"].mean()
by_season.plot(marker="o", figsize=(9, 4), title="Average first-innings score by season")
plt.ylabel("runs"); plt.grid(alpha=0.3); plt.show()

👀 Scores have jumped in recent seasons (the *Impact Player* rule arrived in 2023). **The game changes over time** — remember that when we split our data.

In [ ]:
first_batting = balls[balls["innings"] == 1].groupby("match_id")["batting_team"].first()
m = matches.set_index("match_id").join(first_batting.rename("bat_first"))
print(f"Team batting first won {(m['winner'] == m['bat_first']).mean():.0%} of matches")
runs_per_over = balls.groupby(["innings", "over"])["runs"].sum() / balls.groupby(["innings", "over"])["match_id"].nunique()
runs_per_over.unstack(0).plot(kind="bar", figsize=(11, 4), title="Average runs in each over")
plt.xlabel("over (0 = first)"); plt.show()

## 5. Building the "match state" at every ball

A model can't read a scorecard. We describe the situation after each ball with numbers — **features**:

| Feature | Meaning |
|---|---|
| `runs` | Runs scored so far in this innings |
| `wickets_left` | 10 − wickets fallen |
| `balls_left` | 120 − legal balls bowled |
| `crr` | Current run rate (runs per over) |
| `runs_needed` | *(chase only)* target − runs |
| `rrr` | *(chase only)* required run rate |

And the **label**: did the batting team go on to win?

⚠️ **Leakage warning:** every feature must be something you'd *actually know at that moment*. We compute everything with **running totals (`cumsum`)**, so row *n* only uses balls 1…*n*.

In [ ]:
b = balls.copy()
g = b.groupby(["match_id", "innings"])
b["ball_no"] = g.cumcount() + 1
b["runs_so_far"] = g["runs"].cumsum()
b["wickets_fallen"] = g["wicket"].cumsum().clip(upper=10)
b["legal_balls"] = g["legal"].cumsum()
b["balls_left"] = (120 - b["legal_balls"]).clip(lower=0)
b["wickets_left"] = 10 - b["wickets_fallen"]
b["crr"] = b["runs_so_far"] * 6 / b["legal_balls"].clip(lower=1)

first_total = b[b["innings"] == 1].groupby("match_id")["runs"].sum()
b["target"] = b["match_id"].map(first_total) + 1
b["runs_needed"] = (b["target"] - b["runs_so_far"]).clip(lower=0)
b["rrr"] = b["runs_needed"] * 6 / b["balls_left"].clip(lower=1)

b = b.merge(matches[["match_id", "winner", "season", "date"]], on="match_id")
b["batting_team_won"] = (b["batting_team"] == b["winner"]).astype(int)
b[b["innings"] == 2][["match_id", "batting_team", "over", "runs_so_far", "wickets_left",
                      "balls_left", "target", "runs_needed", "rrr", "batting_team_won"]].head(8)

## 6. Train/test split — by **season**, not by row

Two traps:
1. **Row-level random split:** ball 57 of a match lands in training and ball 58 in the test set. The model has effectively *seen the answer* for that match → scores look amazing, then fail on new matches.
2. **Mixing past and future:** in real life you predict *future* matches using *past* data.

So we train on older seasons and test on the most recent ones.

In [ ]:
TEST_FROM = b["season"].max() - 2        # last 3 seasons = test
chase = b[(b["innings"] == 2) & (b["balls_left"] > 0) & (b["runs_needed"] > 0)].copy()
first = b[b["innings"] == 1].copy()

CHASE_FEATURES = ["runs_needed", "balls_left", "wickets_left", "rrr", "crr", "target"]
FIRST_FEATURES = ["runs_so_far", "balls_left", "wickets_left", "crr"]

chase_train, chase_test = chase[chase["season"] < TEST_FROM], chase[chase["season"] >= TEST_FROM]
first_train, first_test = first[first["season"] < TEST_FROM], first[first["season"] >= TEST_FROM]
print(f"Train: seasons < {TEST_FROM} ({chase_train['match_id'].nunique()} matches)")
print(f"Test : seasons ≥ {TEST_FROM} ({chase_test['match_id'].nunique()} matches)")

## 7. The chase model

We want **probabilities**, not just yes/no. Two models:
- **Logistic regression** — simple, naturally gives probabilities.
- **Gradient boosting** — can learn curved relationships like "7 wickets down changes everything".

### How to judge probabilities
- **Log loss** — punishes being *confidently wrong*. Lower is better.
- **Brier score** — average squared error between the probability and what happened (0 or 1). Lower is better. Guessing 50% every time scores 0.25.
- **Accuracy** — just for intuition.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score

def report(name, y, p):
    print(f"{name:24s} log loss {log_loss(y, p):.3f} | Brier {brier_score_loss(y, p):.3f} | accuracy {accuracy_score(y, p > 0.5):.1%}")

Xtr, ytr = chase_train[CHASE_FEATURES], chase_train["batting_team_won"]
Xte, yte = chase_test[CHASE_FEATURES], chase_test["batting_team_won"]

report("Always 50%", yte, np.full(len(yte), 0.5))
chase_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
report("Logistic regression", yte, chase_lr.predict_proba(Xte)[:, 1])
chase_gb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=4,
                                          monotonic_cst=[-1, 1, 1, -1, 0, 0],  # common-sense rules, see below
                                          random_state=42).fit(Xtr, ytr)
report("Gradient boosting", yte, chase_gb.predict_proba(Xte)[:, 1])

💡 `monotonic_cst` bakes in cricket common sense: *more runs needed → lower chance* (-1), *more balls or wickets left → higher chance* (+1), *higher required rate → lower chance* (-1). This stops the model learning silly wiggles from noisy data.

### Accuracy at different stages of the chase
Predicting the winner is hard early on and easy near the end — a good model should reflect that.

In [ ]:
chase_test = chase_test.assign(p=chase_gb.predict_proba(Xte)[:, 1])
chase_test["over_bucket"] = pd.cut(120 - chase_test["balls_left"], bins=[0, 36, 72, 96, 120],
                                   labels=["overs 1-6", "7-12", "13-16", "17-20"])
chase_test.groupby("over_bucket", observed=True).apply(
    lambda d: pd.Series({"accuracy": accuracy_score(d["batting_team_won"], d["p"] > 0.5),
                         "Brier": brier_score_loss(d["batting_team_won"], d["p"])})).round(3)

## 8. Are the probabilities honest? — Calibration

If the model says **70%** for 100 situations, the chasing team should win about **70** of them. A **calibration plot** checks this: points on the diagonal = honest probabilities.

In [ ]:
from sklearn.calibration import calibration_curve

plt.figure(figsize=(6, 6))
for name, model in [("Logistic regression", chase_lr), ("Gradient boosting", chase_gb)]:
    frac, mean_p = calibration_curve(yte, model.predict_proba(Xte)[:, 1], n_bins=10)
    plt.plot(mean_p, frac, marker="o", label=name)
plt.plot([0, 1], [0, 1], "k--", label="perfect")
plt.xlabel("predicted win probability"); plt.ylabel("actual win rate")
plt.legend(); plt.title("Calibration — chase model"); plt.show()

## 9. The first-innings model — and a leakage experiment

During the first innings there's no target yet, so we use the current score, wickets and balls left.

Let's also **deliberately leak** to see how dangerous it is. We'll add `final_total` — the score the batting team *ends up* making. It's hugely predictive... but at over 5 **nobody knows it yet**.

In [ ]:
first = first.merge(first_total.rename("final_total"), left_on="match_id", right_index=True)
first_train, first_test = first[first["season"] < TEST_FROM], first[first["season"] >= TEST_FROM]

first_gb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3, random_state=42)
first_gb.fit(first_train[FIRST_FEATURES], first_train["batting_team_won"])
report("Honest first-innings", first_test["batting_team_won"], first_gb.predict_proba(first_test[FIRST_FEATURES])[:, 1])

leaky = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_depth=3, random_state=42)
leaky.fit(first_train[FIRST_FEATURES + ["final_total"]], first_train["batting_team_won"])
report("LEAKY (knows final total)", first_test["batting_team_won"],
       leaky.predict_proba(first_test[FIRST_FEATURES + ["final_total"]])[:, 1])

The leaky model looks far better on paper — and would be **useless in a live match**, because you can't feed it tomorrow's newspaper. Whenever results look too good to be true, **hunt for leakage first**.

### The row-split trap
Now the second kind of leakage: splitting randomly by *row* instead of by *match/season*.

In [ ]:
from sklearn.model_selection import train_test_split

r_tr, r_te = train_test_split(chase, test_size=0.2, random_state=42)   # ❌ random rows
m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=6, random_state=42)
m.fit(r_tr[CHASE_FEATURES].assign(mid=r_tr["match_id"].astype("category").cat.codes),
      r_tr["batting_team_won"])
cats = pd.Categorical(r_te["match_id"], categories=r_tr["match_id"].astype("category").cat.categories)
report("Random row split + match id", r_te["batting_team_won"],
       m.predict_proba(r_te[CHASE_FEATURES].assign(mid=cats.codes))[:, 1])
report("Honest season split", yte, chase_gb.predict_proba(Xte)[:, 1])

With a random row split, the model can memorise *which match* a row came from and look the answer up. Honest splits give lower — but **real** — scores.

## 10. 📈 Win-probability worm for a real match

We glue both models together: first innings → probability the team batting first wins; second innings → 1 − chase probability.

In [ ]:
def match_timeline(match_id):
    rows = b[b["match_id"] == match_id].copy()
    inn1, inn2 = rows[rows["innings"] == 1], rows[rows["innings"] == 2]
    bat_first, bat_second = inn1["batting_team"].iloc[0], inn2["batting_team"].iloc[0]
    p1 = first_gb.predict_proba(inn1[FIRST_FEATURES])[:, 1]
    live = inn2[(inn2["balls_left"] > 0) & (inn2["runs_needed"] > 0)]
    p2 = 1 - chase_gb.predict_proba(live[CHASE_FEATURES])[:, 1]
    x1 = inn1["legal_balls"].values / 6
    x2 = 20 + live["legal_balls"].values / 6
    return bat_first, bat_second, np.r_[x1, x2], np.r_[p1, p2], rows["winner"].iloc[0]

def plot_worm(match_id):
    bf, bs, x, p, winner = match_timeline(match_id)
    date = matches.set_index("match_id").loc[match_id, "date"].date()
    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.fill_between(x, 0.5, p, where=p >= 0.5, color="tab:blue", alpha=0.35, interpolate=True, label=short(bf))
    ax.fill_between(x, p, 0.5, where=p < 0.5, color="tab:orange", alpha=0.35, interpolate=True, label=short(bs))
    ax.plot(x, p, color="black", lw=1)
    ax.axhline(0.5, color="grey", lw=0.8); ax.axvline(20, color="grey", ls="--", lw=0.8)
    ax.set_ylim(0, 1); ax.set_xlim(0, 40)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1]); ax.set_yticklabels([f"{short(bs)} 100%", "75%", "50/50", "75%", f"{short(bf)} 100%"])
    ax.set_xlabel("overs (0-20 first innings, 20-40 chase)")
    ax.set_title(f"{short(bf)} vs {short(bs)} · {date} · won by {short(winner)}")
    ax.legend(loc="upper right"); plt.show()

recent = matches[matches["season"] >= TEST_FROM]
plot_worm(recent["match_id"].iloc[-1])

### Over-by-over commentary

In [ ]:
def commentary(match_id):
    bf, bs, x, p, winner = match_timeline(match_id)
    last_shown = -1
    for xi, pi in zip(x, p):
        over = int(np.floor(xi + 1e-9))
        if over > last_shown and abs(xi - round(xi)) < 1e-9 and xi > 0:
            last_shown = over
            leader, prob = (bf, pi) if pi >= 0.5 else (bs, 1 - pi)
            inn = "1st inns" if xi <= 20 else "chase"
            print(f"after over {over if xi <= 20 else over - 20:2d} ({inn}): {short(leader):>4s} {prob:.0%} to win")
    print("Result:", short(winner), "won")

commentary(recent["match_id"].iloc[-1])

## 11. Ask the model about any situation

In [ ]:
def chase_chance(target, runs, wickets_fallen, overs_bowled):
    balls_bowled = int(overs_bowled) * 6 + round((overs_bowled % 1) * 10)   # 15.3 overs = 93 balls
    state = pd.DataFrame([{
        "runs_needed": target - runs, "balls_left": 120 - balls_bowled,
        "wickets_left": 10 - wickets_fallen,
        "rrr": (target - runs) * 6 / max(1, 120 - balls_bowled),
        "crr": runs * 6 / max(1, balls_bowled), "target": target,
    }])
    return chase_gb.predict_proba(state[CHASE_FEATURES])[0, 1]

print(f"Chasing 190, 120/3 after 14 overs: {chase_chance(190, 120, 3, 14):.0%}")
print(f"Chasing 190, 120/6 after 14 overs: {chase_chance(190, 120, 6, 14):.0%}")
print(f"Chasing 190, 150/3 after 14 overs: {chase_chance(190, 150, 3, 14):.0%}")
print(f"Need 16 off the last over, 7 down: {chase_chance(200, 184, 7, 19):.0%}")

In [ ]:
import joblib
joblib.dump({"chase": chase_gb, "first": first_gb, "chase_features": CHASE_FEATURES,
             "first_features": FIRST_FEATURES}, "ipl_win_models.joblib")
print("Saved ipl_win_models.joblib")

## ✅ What you learned
- Turning **raw nested JSON** into clean tables.
- Building **state features** with running totals so nothing comes from the future.
- **Two kinds of leakage** — future information in features, and bad train/test splits — and how they inflate scores.
- Evaluating **probabilities** with log loss, Brier score and **calibration**.

## ✍️ Try it yourself (make it *your* project)
1. **Venue effect:** add each venue's average first-innings score (computed only from *earlier* seasons — careful, leakage!) as a feature.
2. **Team strength:** add each team's win rate over its previous 10 matches. Again: only *past* matches.
3. **Recent momentum:** runs and wickets in the last 18 balls (`rolling` within each innings).
4. **Pre-match model:** predict the winner before a ball is bowled using toss, venue and team form. How much better than 50% can you get?
5. Build a **Streamlit** app where you type the score and get the win probability.

In [ ]:
# Your code here